# Why doesn't retrieval help?

Per-instance comparison of the reference config (`current`) vs. `+ retrieval`, same judge, same instances.

In [21]:
import json
import re
import sys
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import display
from scipy.stats import binomtest
from sklearn.metrics import f1_score

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "src").is_dir() and (p / "output").is_dir()
)
sys.path.insert(0, str(ROOT / "src"))

from novelty_eval.analysis.artifacts import (
    _extract_exclude_set_from_filtered_report, _extract_instances_path)

# The merges behind the paper's controlled-study figure.
MERGES = {"human-only": "merge_hvh_all_metrics", "human+gen": "merge_vanilla_all_metrics"}
MODELS = ["claude-sonnet-4-5", "claude-opus-4-5", "claude-opus-4-6", "gpt-5.1", "gpt-5.2", "gpt-5.4"]

# Pairwise per-instance score: "strict" (tie = wrong, the paper's main figure) or "soft" (tie = 0.5).
PAIRWISE_METRIC = "strict"

## 0. Load

One row per (track, mode, model, instance). Blocklisted instances are dropped and missing predictions count as wrong, as in the paper.

In [22]:
# <mode>-<track>_<ablation>-<model>_<timestamp>
NAME_RE = re.compile(
    r"^(pairwise|pointwise)-[^_]+_(current|retrieval)-(.+)_\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}$")

DIRS = {}  # (track, mode, model, cond) -> artifact dir
for track, merge in MERGES.items():
    for d in sorted((ROOT / "output/ablation_sweeps" / merge / "artifacts").iterdir()):
        if m := NAME_RE.match(d.name):
            key = (track, m[1], m[3], m[2])
            assert key not in DIRS, f"two runs for {key}"
            DIRS[key] = d


def load_run(d, mode):
    """gold/pred: pointwise 1 = novel; pairwise = position (0/1) of the idea in the comparison, pred 2 = tie."""
    instances = yaml.safe_load((ROOT / _extract_instances_path(d)).read_text())
    scores = json.loads(next(d.glob("run_*/scores.json")).read_text())
    excluded = _extract_exclude_set_from_filtered_report(d)
    rows = []
    for iid, inst in instances.items():
        iid = str(iid)
        if iid in excluded:
            continue
        s = scores.get(iid)
        if mode == "pointwise":
            gold = int(inst["label"] == "POSITIVE")
            pred = None if s is None else int(s["prediction"])
        else:
            comp = s["comparisons"][0] if s else None
            winners = {str(w) for w in inst["expected_winners"]}
            gold = None if comp is None else int(str(comp["idea_0"]) not in winners)
            pred = None if comp is None else comp["winner"]
        rows.append({"id": iid, "gold": gold, "pred": pred, "ok": pred is not None and pred == gold, "tie": pred == 2})
    return pd.DataFrame(rows).set_index("id")


def macro_f1(gold, pred):
    """Missing predictions count as wrong, as in the paper."""
    pred = pd.to_numeric(pred).fillna(1 - gold)
    return f1_score(gold.astype(int), pred.astype(int), average="macro")


RUNS = {k: load_run(d, k[1]) for k, d in DIRS.items()}

# Must reproduce the filtered reports the paper's figures are built from.
CHECKS = {
    "pointwise": [(r"Mean F1 \(macro\):", lambda r: macro_f1(r.gold, r.pred))],
    "pairwise": [(r"Mean LLM Pairwise Accuracy \(strict\):", lambda r: r.ok.mean()),
                 (r"Mean LLM Pairwise Accuracy \(with ties\):", lambda r: (r.ok + 0.5 * r.tie).mean())],
}
for key, r in RUNS.items():
    report = (DIRS[key] / "filtered_accuracy_report.txt").read_text()
    for pat, metric in CHECKS[key[1]]:
        reported = float(re.search(pat + r"\s+([\d.]+)", report)[1])
        assert abs(metric(r) - reported) < 1e-3, (key, pat, metric(r), reported)
print(f"{len(RUNS)} runs, all match their filtered reports")

48 runs, all match their filtered reports


In [23]:
# Paired view: one row per instance, reference vs. retrieval side by side.
frames = []
for track, mode, model in sorted({k[:3] for k in RUNS}):
    ref, ret = RUNS[track, mode, model, "current"], RUNS[track, mode, model, "retrieval"]
    assert ref.index.equals(ret.index), (track, mode, model)
    frames.append(pd.DataFrame({
        "track": track, "mode": mode, "model": model,
        "gold": ref.gold.fillna(ret.gold),
        "ref_pred": ref.pred, "ret_pred": ret.pred,
        "ref_ok": ref.ok, "ret_ok": ret.ok,
        "ref_tie": ref.tie, "ret_tie": ret.tie,
    }))
PAIRED = pd.concat(frames).reset_index()
PAIRED.groupby(["track", "mode"]).size().unstack()

mode,pairwise,pointwise
track,,
human+gen,924,1848
human-only,924,1794


## 1. Helped vs. hurt

helped / hurt = the per-instance score went up / down with retrieval. `p`: sign test on helped vs. hurt (= exact McNemar for 0/1 scores).
Pointwise: macro-F1, flips = instance correctness. Pairwise score is set by `PAIRWISE_METRIC`.

In [24]:
# Score-changing pairwise transitions to break the flips into, per metric.
TRANSITIONS = {
    "strict": ["✗→✓", "tie→✓", "✓→✗", "✓→tie"],
    "soft": ["✗→✓", "tie→✓", "✗→tie", "✓→✗", "✓→tie", "tie→✗"],
}
PAIRWISE_NAMES = {
    "strict": "strict accuracy (tie = wrong)",
    "soft": "soft accuracy (tie = 0.5)",
}


def flip_stats(g):
    pointwise = g["mode"].iat[0] == "pointwise"
    credit = 0.5 if PAIRWISE_METRIC == "soft" else 0.0  # pointwise has no ties
    ref, ret = g.ref_ok + credit * g.ref_tie, g.ret_ok + credit * g.ret_tie
    helped, hurt = ret > ref, ret < ref
    n_flips = helped.sum() + hurt.sum()

    if pointwise:
        f1_ref, f1_ret = macro_f1(g.gold, g.ref_pred), macro_f1(g.gold, g.ret_pred)
        row = {"n": len(g), "F1_ref": 100 * f1_ref, "ΔF1": 100 * (f1_ret - f1_ref)}
    else:
        name = PAIRWISE_METRIC
        row = {"n": len(g), f"{name}_ref": 100 * ref.mean(), f"Δ{name}": 100 * (ret.mean() - ref.mean())}
    row |= {
        "helped": helped.sum(), "hurt": hurt.sum(), "net": helped.sum() - hurt.sum(),
        "both ✓": ((ref == 1) & (ret == 1)).sum(), "both ✗": ((ref == 0) & (ret == 0)).sum(),
        "p": binomtest(helped.sum(), n_flips).pvalue if n_flips else 1.0,
    }
    if pointwise:
        # Which class the flips land on: D+ = novel (gold 1), D- = lower-novelty (gold 0).
        for cls, cls_name in [(1, "D+"), (0, "D-")]:
            row[f"helped {cls_name}"] = (helped & (g.gold == cls)).sum()
            row[f"hurt {cls_name}"] = (hurt & (g.gold == cls)).sum()
    else:
        # Whether the flips are real verdict changes or moves in and out of a tie.
        state = lambda ok, tie: pd.Series("✗", g.index).mask(ok, "✓").mask(tie, "tie")
        trans = (state(g.ref_ok, g.ref_tie) + "→" + state(g.ret_ok, g.ret_tie))[helped | hurt]
        row |= {t: (trans == t).sum() for t in TRANSITIONS[PAIRWISE_METRIC]}
        row["tie% ref→ret"] = f"{100 * g.ref_tie.mean():.0f}→{100 * g.ret_tie.mean():.0f}"
    return pd.Series(row)


def flip_table(mode):
    t = (PAIRED[PAIRED["mode"] == mode]
         .groupby(["track", "model"]).apply(flip_stats, include_groups=False)
         .reindex(pd.MultiIndex.from_product([MERGES, MODELS], names=["track", "model"])))
    fmt = {c: "{:+.1f}" for c in t.columns if c.startswith("Δ")}
    fmt |= {c: "{:.1f}" for c in t.columns if c.endswith("_ref")} | {"p": "{:.3f}"}
    caption = {
        "pointwise": "Pointwise · macro-F1 (paper's metric) · helped/hurt = instance flipped ✗→✓ / ✓→✗",
        "pairwise": f"Pairwise · helped/hurt on {PAIRWISE_NAMES[PAIRWISE_METRIC]}",
    }[mode]
    return t.style.format(fmt, precision=0).set_caption(caption)


display(flip_table("pointwise"))
display(flip_table("pairwise"))